In [ ]:
from __future__ import annotations

import math
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageFile
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True

# -----------------------------------------------------------------------------
# 0. Configuration
# -----------------------------------------------------------------------------

SEED = 42
IMAGE_SIZE = 256
BATCH_SIZE = 16
NUM_WORKERS = 8
N_THRESHOLD_CANDIDATES = 181  # matches Section III.E: sweep [0.05, 0.95]

# >>> Confirm this is correct for your standardized 70/10/20 dataset. <<<
DATASET_HANDLE = "nocturnalnerd/balanced-dataset-701020"

# >>> Point this at wherever manifest.csv actually lives. <<<
# Defaults to DATASET_ROOT / "manifest.csv" -- override if it's elsewhere
# (e.g. uploaded separately into /kaggle/working).
MANIFEST_PATH_OVERRIDE: Path | None = None  # e.g. Path("/kaggle/working/manifest.csv")

# Values the script will accept for the "split" column, case/whitespace
# insensitive. Edit these if the printed unique values at startup don't match.
VALIDATION_SPLIT_VALUES = {"validation", "val", "valid"}
TEST_SPLIT_VALUES = {"test"}

# >>> Point these at your four trained checkpoints. <<<
ORIGINAL_FREQNET_CHECKPOINT_PATH = Path("/kaggle/input/datasets/mikekomari/models/best_freqnet_rgb_baseline (1).pt")
SIMPLE_CNN_CHECKPOINT_PATH = Path("/kaggle/input/datasets/mikekomari/models/normalCNN.pt")
ALEXNET_CHECKPOINT_PATH = Path("/kaggle/input/datasets/mikekomari/models/best_alexnet_scratch.pt")
# Dynamic FreqNet retrained with the ORIGINAL FreqNet recipe (DynamicFreqnet_matched.py, INPUT_MODE="dynamic").
DYNAMIC_CHECKPOINT_PATH = Path("/kaggle/input/datasets/mikeeee54/dynamiccc/best_freqnet_dynamic_c0.20_seed42 (2).pt")

REPO_DIR = Path("/kaggle/working/FreqNet-DeepfakeDetection")
REPO_URL = "https://github.com/chuangchuangtan/FreqNet-DeepfakeDetection.git"

OUTPUT_DIR = Path("/kaggle/working/four_model_evaluation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)
print("Device:", device)

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)


def resolve_dataset_root() -> Path:
    explicit_root = os.environ.get("DATASET_ROOT")
    if explicit_root:
        explicit_path = Path(explicit_root)
        if explicit_path.exists():
            return explicit_path
        raise FileNotFoundError(f"DATASET_ROOT={explicit_path} does not exist.")
    try:
        import kagglehub
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
        import kagglehub
    downloaded_root = Path(kagglehub.dataset_download(DATASET_HANDLE))
    direct = downloaded_root / "dataset_result_70_10_20_balanced"
    if direct.exists():
        return direct
    discovered = [p for p in downloaded_root.rglob("dataset_result_70_10_20_balanced") if p.is_dir()]
    if len(discovered) == 1:
        return discovered[0]
    if all((downloaded_root / split).is_dir() for split in ("train", "validation", "test")):
        return downloaded_root
    raise FileNotFoundError(f"Could not locate dataset_result_balanced under {downloaded_root}.")


DATASET_ROOT = resolve_dataset_root()
MANIFEST_PATH = MANIFEST_PATH_OVERRIDE or (DATASET_ROOT / "manifest.csv")
print("Dataset root:", DATASET_ROOT)
print("Manifest path:", MANIFEST_PATH)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
freqnet_source = REPO_DIR / "networks" / "freqnet.py"
source_text = freqnet_source.read_text(encoding="utf-8")
patched_text = source_text.replace(".cuda()", "")
if patched_text != source_text:
    freqnet_source.write_text(patched_text, encoding="utf-8")
sys.path.insert(0, str(REPO_DIR))
from networks.freqnet import FreqNet  # noqa: E402


# -----------------------------------------------------------------------------
# 1. Manifest loading -- fails loudly with diagnostics if assumptions are wrong
# -----------------------------------------------------------------------------


def normalize_label(value) -> int:
    if isinstance(value, (int, float)) and not isinstance(value, bool) and not pd.isna(value):
        if int(value) in (0, 1):
            return int(value)
    text = str(value).strip().lower()
    if text in {"0", "real", "authentic", "genuine"}:
        return 0
    if text in {"1", "fake", "synthetic", "generated", "ai", "ai-generated", "ai_generated"}:
        return 1
    raise ValueError(
        f"Unrecognized label value: {value!r}. Edit normalize_label() in this script to handle it, "
        f"or tell me the unique label values printed above and I'll update it for you."
    )


def load_and_validate_manifest(manifest_path: Path, dataset_root: Path) -> pd.DataFrame:
    if not manifest_path.exists():
        raise FileNotFoundError(f"manifest.csv not found at {manifest_path}. Set MANIFEST_PATH_OVERRIDE.")

    df = pd.read_csv(manifest_path)
    print(f"\nLoaded manifest: {len(df):,} rows, columns: {list(df.columns)}")

    required_cols = {"split", "label", "relative_path", "source_model", "source_dataset"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"manifest.csv is missing expected columns: {missing}. Found: {list(df.columns)}")

    print("Unique 'split' values:", df["split"].astype(str).str.strip().unique().tolist())
    print("Unique 'label' values:", df["label"].astype(str).str.strip().unique().tolist()[:10])
    print("Unique 'source_dataset' values (sample):", df["source_dataset"].astype(str).unique().tolist()[:10])
    print("Unique 'source_model' values (sample):", df["source_model"].astype(str).unique().tolist()[:15])

    df["split_norm"] = df["split"].astype(str).str.strip().str.lower()
    df["label_binary"] = df["label"].apply(normalize_label)

    sample_n = min(200, len(df))
    sample = df.sample(sample_n, random_state=SEED)
    exists_mask = sample["relative_path"].apply(lambda p: (dataset_root / str(p)).exists())
    hit_rate = exists_mask.mean()
    print(f"\nPath check: {int(exists_mask.sum())}/{sample_n} sampled relative_path values "
          f"resolve under {dataset_root} ({hit_rate:.1%})")

    if hit_rate < 0.95:
        print("\nExample relative_path values that did NOT resolve:")
        for p in sample.loc[~exists_mask, "relative_path"].head(5):
            print(" ", p, "->", dataset_root / str(p))
        if "original_path" in df.columns:
            print("\nFor reference, 'original_path' for those same rows:")
            print(sample.loc[~exists_mask, "original_path"].head(5).to_list())
        raise FileNotFoundError(
            "Fewer than 95% of sampled relative_path values resolve under DATASET_ROOT. "
            "relative_path is probably relative to something other than DATASET_ROOT, or "
            "DATASET_ROOT itself is wrong. Send me a few example rows (relative_path, "
            "original_path, split) and I will fix the join logic before you run this for real."
        )

    return df


manifest_df = load_and_validate_manifest(MANIFEST_PATH, DATASET_ROOT)

val_df = manifest_df[manifest_df["split_norm"].isin(VALIDATION_SPLIT_VALUES)].reset_index(drop=True)
test_df = manifest_df[manifest_df["split_norm"].isin(TEST_SPLIT_VALUES)].reset_index(drop=True)
print(f"\nValidation rows matched: {len(val_df):,}  |  Test rows matched: {len(test_df):,}")
if len(val_df) == 0 or len(test_df) == 0:
    raise ValueError(
        "Matched 0 rows for validation or test. VALIDATION_SPLIT_VALUES / TEST_SPLIT_VALUES don't "
        "match the unique 'split' values printed above -- update them and rerun."
    )


# -----------------------------------------------------------------------------
# 2. Manifest-driven dataset -- keeps source_model / source_dataset / path
#    alongside each image so results can be grouped by generator afterward.
# -----------------------------------------------------------------------------


class ManifestImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame, image_root: Path, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_root = Path(image_root)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        image = Image.open(self.image_root / str(row["relative_path"])).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(float(row["label_binary"]), dtype=torch.float32)
        return image, label, str(row["source_model"]), str(row["source_dataset"]), str(row["relative_path"])


eval_transform = transforms.Compose(
    [transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True), transforms.ToTensor()]
)

val_dataset = ManifestImageDataset(val_df, DATASET_ROOT, eval_transform)
test_dataset = ManifestImageDataset(test_df, DATASET_ROOT, eval_transform)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))


# -----------------------------------------------------------------------------
# 3. Model classes -- copied verbatim from your training scripts so checkpoint
#    state_dict keys line up exactly.
# -----------------------------------------------------------------------------

# Must match the training run that produced DYNAMIC_CHECKPOINT_PATH.
MID_HIGH_CUTOFF = 0.20
NUM_BANDS = 8
SELECTOR_HIDDEN = 32

HFRI_CUTOFF = 0.05
WMBFR_CUTOFF = 0.10


class MultiBandFrequencyFusion(nn.Module):
    def __init__(self, hfri_cutoff: float = HFRI_CUTOFF, wmbfr_cutoff: float = WMBFR_CUTOFF, eps: float = 1e-6):
        super().__init__()
        self.hfri_cutoff = hfri_cutoff
        self.wmbfr_cutoff = wmbfr_cutoff
        self.eps = eps

    def _standardize(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=(-2, -1), keepdim=True)
        std = x.std(dim=(-2, -1), keepdim=True).clamp_min(self.eps)
        return (x - mean) / std

    def _radius(self, h: int, w: int, x: torch.Tensor) -> torch.Tensor:
        yy = torch.linspace(-1.0, 1.0, h, device=x.device, dtype=x.dtype)
        xx = torch.linspace(-1.0, 1.0, w, device=x.device, dtype=x.dtype)
        grid_y, grid_x = torch.meshgrid(yy, xx, indexing="ij")
        radius = torch.sqrt(grid_x.square() + grid_y.square())
        radius = radius / radius.max().clamp_min(self.eps)
        return radius[None, None, :, :]

    def forward(self, rgb: torch.Tensor) -> torch.Tensor:
        if rgb.ndim != 4 or rgb.shape[1] != 3:
            raise ValueError(f"Expected [B,3,H,W] RGB input; got {tuple(rgb.shape)}")
        with torch.amp.autocast(device_type=rgb.device.type, enabled=False):
            rgb = rgb.float()
            gray = 0.2989 * rgb[:, 0:1] + 0.5870 * rgb[:, 1:2] + 0.1140 * rgb[:, 2:3]
            spectrum = torch.fft.fftshift(torch.fft.fft2(gray, dim=(-2, -1), norm="ortho"), dim=(-2, -1))
            radius = self._radius(gray.shape[-2], gray.shape[-1], gray)

            hfri_mask = (radius > self.hfri_cutoff).to(gray.dtype)
            hfri = torch.fft.ifft2(
                torch.fft.ifftshift(spectrum * hfri_mask, dim=(-2, -1)), dim=(-2, -1), norm="ortho"
            ).abs()

            k_f = -0.2 * radius.square() + 0.8 * radius - 0.05
            weighted = (torch.exp(k_f / 2.0) - 1.0) / radius.clamp_min(self.eps)
            wmbfr_mask = torch.where(radius > self.wmbfr_cutoff, weighted, torch.zeros_like(weighted))
            wmbfr = torch.fft.ifft2(
                torch.fft.ifftshift(spectrum * wmbfr_mask, dim=(-2, -1)), dim=(-2, -1), norm="ortho"
            ).abs()

            return torch.cat([self._standardize(gray), self._standardize(hfri), self._standardize(wmbfr)], dim=1)


class AmpSafeFreqNet(FreqNet):
    def hfreqWH(self, x: torch.Tensor, scale: int) -> torch.Tensor:
        assert scale > 2
        x = x.float()
        x = torch.fft.fft2(x, norm="ortho")
        x = torch.fft.fftshift(x, dim=[-2, -1])
        _, _, h, w = x.shape
        mask = torch.ones((h, w), device=x.device, dtype=x.real.dtype)
        mask[h // 2 - h // scale : h // 2 + h // scale, w // 2 - w // scale : w // 2 + w // scale] = 0.0
        x = x * mask[None, None, :, :]
        x = torch.fft.ifftshift(x, dim=[-2, -1])
        x = torch.fft.ifft2(x, norm="ortho").real
        return F.relu(x, inplace=False)

    def hfreqC(self, x: torch.Tensor, scale: int) -> torch.Tensor:
        assert scale > 2
        x = x.float()
        x = torch.fft.fft(x, dim=1, norm="ortho")
        x = torch.fft.fftshift(x, dim=1)
        _, c, _, _ = x.shape
        mask = torch.ones(c, device=x.device, dtype=x.real.dtype)
        mask[c // 2 - c // scale : c // 2 + c // scale] = 0.0
        x = x * mask[None, :, None, None]
        x = torch.fft.ifftshift(x, dim=1)
        x = torch.fft.ifft(x, dim=1, norm="ortho").real
        return F.relu(x, inplace=False)

    def _frequency_conv(self, x: torch.Tensor, real_conv: nn.Module, imag_conv: nn.Module) -> torch.Tensor:
        x = x.float()
        x = torch.fft.fft2(x, norm="ortho")
        x = torch.fft.fftshift(x, dim=[-2, -1])
        x = torch.complex(real_conv(x.real), imag_conv(x.imag))
        x = torch.fft.ifftshift(x, dim=[-2, -1])
        x = torch.fft.ifft2(x, norm="ortho").real
        return F.relu(x, inplace=False)

    def _raw_projection(self, x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor, stride: int) -> torch.Tensor:
        x = F.conv2d(x.float(), weight.float(), bias.float(), stride=stride, padding=0)
        return F.relu(x, inplace=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.hfreqWH(x, 4)
        x = self._raw_projection(x, self.weight1, self.bias1, stride=1)
        x = self.hfreqC(x, 4)
        x = self._frequency_conv(x, self.realconv1, self.imagconv1)

        x = self.hfreqWH(x, 4)
        x = self._raw_projection(x, self.weight2, self.bias2, stride=2)
        x = self.hfreqC(x, 4)
        x = self._frequency_conv(x, self.realconv2, self.imagconv2)

        x = self.maxpool(x)
        x = self.layer1(x)

        x = self.hfreqWH(x, 4)
        x = self._raw_projection(x, self.weight3, self.bias3, stride=1)
        x = self._frequency_conv(x, self.realconv3, self.imagconv3)

        x = self.hfreqWH(x, 4)
        x = self._raw_projection(x, self.weight4, self.bias4, stride=2)
        x = self._frequency_conv(x, self.realconv4, self.imagconv4)

        x = self.layer2(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        return self.fc1(x)


class DynamicAmpSafeFreqNet(AmpSafeFreqNet):
    """AmpSafeFreqNet whose FIRST frequency operation is replaced. Nothing else changes.

    The original first step (hfreqWH, scale=4) is
        FFT -> zero a fixed central square -> IFFT -> ReLU.
    Here it is
        FFT -> multiply by a band mask -> IFFT -> ReLU,
    with the FFT / shift / IFFT / real-part / ReLU / FP32 handling copied line by
    line from AmpSafeFreqNet.hfreqWH. Only the mask differs:

      "dynamic" : one weight per radial band, predicted per image from the
                  within-image-normalised mean log-magnitude of each band (MLP).
      "static"  : same rings, one learned weight per band shared by all images.
      "ones"    : same rings, every weight = 1 (a plain radial high-pass).
      "freqnet" : the original fixed square mask, run through this code path
                  (used only by the regression check below).

    Every later layer (hfreqC, frequency convs, residual layers, later hfreqWH
    calls) is inherited unchanged. super().__init__() runs first, so the backbone
    weights get the same RNG stream -- and therefore the same initial values --
    as the original model for a given seed.
    """

    MODES = ("dynamic", "static", "ones", "freqnet")

    def __init__(self, first_stage_mode: str = "dynamic", cutoff: float = 0.20, num_bands: int = 8,
                 hidden_dim: int = 32, image_size: int = 256):
        super().__init__()
        if first_stage_mode not in self.MODES:
            raise ValueError(f"first_stage_mode must be one of {self.MODES}")
        self.first_stage_mode = first_stage_mode
        self.cutoff = float(cutoff)
        self.num_bands = int(num_bands)
        self._stem_pending = False

        if first_stage_mode == "dynamic":
            self.dynamic_selector = nn.Sequential(
                nn.Linear(num_bands, hidden_dim), nn.ReLU(inplace=True),
                nn.Linear(hidden_dim, num_bands), nn.Sigmoid(),
            )
        elif first_stage_mode == "static":
            self.static_band_logits = nn.Parameter(torch.zeros(num_bands))  # sigmoid(0) = 0.5, like the MLP at init

        band_plus1, membership, counts = self._build_band_geometry(image_size, image_size)
        self.register_buffer("_band_plus1", band_plus1, persistent=False)   # 0 = suppressed, k+1 = band k
        self.register_buffer("_membership", membership, persistent=False)   # (H*W, K) one-hot
        self.register_buffer("_band_counts", counts, persistent=False)      # (K,)

    # -- band geometry (paper Eqs. 5-7) ------------------------------------------
    def _build_band_geometry(self, h: int, w: int):
        yy = torch.linspace(-1.0, 1.0, h)
        xx = torch.linspace(-1.0, 1.0, w)
        grid_y, grid_x = torch.meshgrid(yy, xx, indexing="ij")
        rho = torch.sqrt(grid_x.square() + grid_y.square())
        rho = rho / rho.max()                                                # normalised radius in [0, 1]
        edges = torch.linspace(self.cutoff, 1.0, self.num_bands + 1)
        band = torch.bucketize(rho, edges[1:-1], right=True)                 # 0..K-1; last band is closed at 1.0
        band = torch.where(rho >= self.cutoff, band, torch.full_like(band, -1))
        band_plus1 = (band + 1).flatten()
        membership = F.one_hot(band_plus1, self.num_bands + 1)[:, 1:].float()
        counts = membership.sum(dim=0)
        if bool((counts == 0).any()):
            raise ValueError("At least one radial band is empty; lower NUM_BANDS or the cutoff.")
        return band_plus1, membership, counts

    # -- per-image descriptor and weights (paper Eqs. 8-10) -----------------------
    def _band_descriptor(self, spectrum: torch.Tensor) -> torch.Tensor:
        h, w = spectrum.shape[-2:]
        # spectrum is orthonormal-FFT; * sqrt(H*W) recovers the plain DFT magnitude used in Eq. 3-4.
        log_mag = torch.log1p(spectrum.abs() * math.sqrt(h * w)).mean(dim=1)      # mean over channels -> (B, H, W)
        per_band = log_mag.flatten(1) @ self._membership / self._band_counts      # (B, K)
        return per_band / (per_band.mean(dim=1, keepdim=True) + 1e-6)             # within-image normalisation

    def _band_weights(self, spectrum: torch.Tensor) -> torch.Tensor:
        batch = spectrum.shape[0]
        if self.first_stage_mode == "dynamic":
            return self.dynamic_selector(self._band_descriptor(spectrum))
        if self.first_stage_mode == "static":
            return torch.sigmoid(self.static_band_logits).unsqueeze(0).expand(batch, -1)
        return torch.ones(batch, self.num_bands, device=spectrum.device, dtype=spectrum.real.dtype)

    def _band_mask(self, weights: torch.Tensor, h: int, w: int) -> torch.Tensor:
        padded = torch.cat([weights.new_zeros(weights.shape[0], 1), weights], dim=1)  # column 0 -> suppressed
        return padded[:, self._band_plus1].reshape(-1, 1, h, w)

    @torch.no_grad()
    def inspect_stem(self, x: torch.Tensor):
        """(descriptor, weights), both (B, K), for a normalised RGB batch -- for diagnostics only."""
        with torch.amp.autocast(device_type=x.device.type, enabled=False):
            spectrum = torch.fft.fftshift(torch.fft.fft2(x.float(), norm="ortho"), dim=[-2, -1])
            return self._band_descriptor(spectrum), self._band_weights(spectrum)

    # -- the ONLY replaced operation ----------------------------------------------
    def _first_stage(self, x: torch.Tensor, scale: int) -> torch.Tensor:
        with torch.amp.autocast(device_type=x.device.type, enabled=False):
            x = x.float()
            x = torch.fft.fft2(x, norm="ortho")
            x = torch.fft.fftshift(x, dim=[-2, -1])
            _, _, h, w = x.shape
            if self.first_stage_mode == "freqnet":
                mask = torch.ones((h, w), device=x.device, dtype=x.real.dtype)
                mask[h // 2 - h // scale : h // 2 + h // scale, w // 2 - w // scale : w // 2 + w // scale] = 0.0
                mask = mask[None, None, :, :]
            else:
                mask = self._band_mask(self._band_weights(x), h, w)
            x = x * mask
            x = torch.fft.ifftshift(x, dim=[-2, -1])
            x = torch.fft.ifft2(x, norm="ortho").real
            return F.relu(x, inplace=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self._stem_pending = True   # the next hfreqWH call is the first, image-level one
        return super().forward(x)

    def hfreqWH(self, x: torch.Tensor, scale: int) -> torch.Tensor:
        if self._stem_pending:
            self._stem_pending = False
            if x.shape[1] != 3:
                raise RuntimeError(f"Expected the 3-channel image at the first stage; got {tuple(x.shape)}")
            return self._first_stage(x, scale)
        return super().hfreqWH(x, scale)


FIRST_STAGE_FOR_MODE = {"dynamic": "dynamic", "static_ring": "static", "ring_ones": "ones"}


class ExperimentModel(nn.Module):
    def __init__(self, input_mode: str):
        super().__init__()
        if input_mode != "rgb_baseline" and input_mode not in FIRST_STAGE_FOR_MODE:
            raise ValueError(f"input_mode must be 'rgb_baseline' or one of {sorted(FIRST_STAGE_FOR_MODE)}")
        self.input_mode = input_mode
        if input_mode == "rgb_baseline":
            self.backbone = AmpSafeFreqNet()
        else:
            self.backbone = DynamicAmpSafeFreqNet(
                first_stage_mode=FIRST_STAGE_FOR_MODE[input_mode], cutoff=MID_HIGH_CUTOFF,
                num_bands=NUM_BANDS, hidden_dim=SELECTOR_HIDDEN, image_size=IMAGE_SIZE,
            )
        self.register_buffer("imagenet_mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("imagenet_std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, rgb: torch.Tensor) -> torch.Tensor:
        model_input = (rgb - self.imagenet_mean) / self.imagenet_std
        return self.backbone(model_input).flatten()


class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        def block(in_ch: int, out_ch: int, pool: bool = True) -> nn.Sequential:
            layers = [nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)]
            if pool:
                layers.append(nn.MaxPool2d(2))
            return nn.Sequential(*layers)

        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 256), block(256, 256, pool=False))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(256, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x).flatten()


class AlexNetScratch(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=2), nn.BatchNorm2d(96), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(96, 256, kernel_size=5, padding=2), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(256, 384, kernel_size=3, padding=1), nn.BatchNorm2d(384), nn.ReLU(inplace=True),
            nn.Conv2d(384, 384, kernel_size=3, padding=1), nn.BatchNorm2d(384), nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.pool = nn.AdaptiveAvgPool2d((6, 6))
        self.classifier = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(256 * 6 * 6, 4096), nn.ReLU(inplace=True),
            nn.Dropout(0.5), nn.Linear(4096, 4096), nn.ReLU(inplace=True),
            nn.Linear(4096, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x).flatten()


MODEL_KINDS = ["original_freqnet", "simple_cnn", "alexnet_scratch", "dynamic_freqnet"]
CHECKPOINTS = {
    "original_freqnet": ORIGINAL_FREQNET_CHECKPOINT_PATH,
    "simple_cnn": SIMPLE_CNN_CHECKPOINT_PATH,
    "alexnet_scratch": ALEXNET_CHECKPOINT_PATH,
    "dynamic_freqnet": DYNAMIC_CHECKPOINT_PATH
}
    # "proposed": PROPOSED_CHECKPOINT_PATH,


def build_model(kind: str) -> nn.Module:
    if kind == "original_freqnet":
        return ExperimentModel(input_mode="rgb_baseline")
    if kind == "simple_cnn":
        return SimpleCNN()
    if kind == "alexnet_scratch":
        return AlexNetScratch()
    if kind == "dynamic_freqnet":
        return ExperimentModel(input_mode="dynamic")
    raise ValueError(f"Unknown model kind: {kind!r}")


def load_checkpoint_into(model: nn.Module, path: Path) -> nn.Module:
    if not path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {path}")
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    state_dict = checkpoint.get("model_state_dict", checkpoint) if isinstance(checkpoint, dict) else checkpoint
    state_dict = {k[len("module.") :] if k.startswith("module.") else k: v for k, v in state_dict.items()}
    model.load_state_dict(state_dict, strict=True)
    return model


# -----------------------------------------------------------------------------
# 4. Inference, threshold selection, metrics
# -----------------------------------------------------------------------------


@torch.inference_mode()
def run_inference(model: nn.Module, kind: str, loader: DataLoader):
    model.eval()
    y_true, y_prob, source_models, source_datasets, paths = [], [], [], [], []
    needs_manual_normalize = kind in {"simple_cnn", "alexnet_scratch"}

    for images, labels, s_model, s_dataset, rel_path in tqdm(loader, desc=f"infer[{kind}]", leave=False):
        images = images.to(device, non_blocking=True)
        if needs_manual_normalize:
            images = (images - IMAGENET_MEAN) / IMAGENET_STD
        logits = model(images)
        y_prob.append(torch.sigmoid(logits).cpu().numpy())
        y_true.append(labels.numpy().astype(np.int64))
        source_models.extend(s_model)
        source_datasets.extend(s_dataset)
        paths.extend(rel_path)

    return (
        np.concatenate(y_true),
        np.concatenate(y_prob),
        np.array(source_models),
        np.array(source_datasets),
        np.array(paths),
    )


def select_f1_threshold(y_true: np.ndarray, y_prob: np.ndarray, n_candidates: int = N_THRESHOLD_CANDIDATES):
    candidates = np.linspace(0.05, 0.95, n_candidates)
    best_t, best_f1 = 0.5, -1.0
    for t in candidates:
        f1 = f1_score(y_true, (y_prob >= t).astype(np.int64), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    return best_t, best_f1


def full_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> dict:
    y_pred = (y_prob >= threshold).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    real_mask, fake_mask = (y_true == 0), (y_true == 1)
    return {
        "n": int(len(y_true)),
        "threshold": threshold,
        "roc_auc": roc_auc_score(y_true, y_prob),
        "average_precision": average_precision_score(y_true, y_prob),
        "brier_score": brier_score_loss(y_true, y_prob),
        "log_loss": log_loss(y_true, y_prob, labels=[0, 1]),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "real_accuracy": float(accuracy_score(y_true[real_mask], y_pred[real_mask])) if real_mask.any() else np.nan,
        "fake_accuracy": float(accuracy_score(y_true[fake_mask], y_pred[fake_mask])) if fake_mask.any() else np.nan,
        "precision_fake": precision_score(y_true, y_pred, zero_division=0),
        "recall_fake": recall_score(y_true, y_pred, zero_division=0),
        "specificity_real": tn / max(tn + fp, 1),
        "f1_fake": f1_score(y_true, y_pred, zero_division=0),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def wilson_ci(k: int, n: int, z: float = 1.96):
    """95% Wilson score interval for a binomial proportion k/n -- appropriate
    for small-n generators where a normal approximation would be misleading."""
    if n == 0:
        return (np.nan, np.nan, np.nan)
    p_hat = k / n
    denom = 1 + z**2 / n
    center = (p_hat + z**2 / (2 * n)) / denom
    half_width = (z * np.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2))) / denom
    return p_hat, max(0.0, center - half_width), min(1.0, center + half_width)


def per_generator_table(y_true: np.ndarray, y_prob: np.ndarray, source_model: np.ndarray, threshold: float) -> pd.DataFrame:
    """One row per generator. accuracy/AP follow Tan et al.'s own convention
    (all real test images + that generator's fake images). detection_rate is
    computed on that generator's fake images alone (no reals mixed in) --
    the metric your notes already track informally (e.g. ~39% InfoMaxGAN vs
    ~96% DALL-E 3), reported here with a Wilson 95% CI.
    """
    y_pred = (y_prob >= threshold).astype(np.int64)
    real_mask = y_true == 0
    fake_generators = sorted(set(source_model[y_true == 1]))

    rows = []
    for gen in fake_generators:
        gen_fake_mask = (y_true == 1) & (source_model == gen)
        combined_mask = real_mask | gen_fake_mask

        n_gen = int(gen_fake_mask.sum())
        k_gen = int((y_pred[gen_fake_mask] == 1).sum())
        detection_rate, ci_lo, ci_hi = wilson_ci(k_gen, n_gen)

        rows.append({
            "generator": gen,
            "n_fake_images": n_gen,
            "accuracy_with_reals": accuracy_score(y_true[combined_mask], y_pred[combined_mask]),
            "average_precision_with_reals": average_precision_score(y_true[combined_mask], y_prob[combined_mask]),
            "detection_rate": detection_rate,
            "detection_rate_ci_lo": ci_lo,
            "detection_rate_ci_hi": ci_hi,
        })
    return pd.DataFrame(rows)


# -----------------------------------------------------------------------------
# 5. Run all four models
# -----------------------------------------------------------------------------

overall_rows = []
per_generator_frames = []
long_prediction_frames = []

for kind in MODEL_KINDS:
    print(f"\n{'=' * 70}\n{kind}\n{'=' * 70}")
    model = build_model(kind).to(device)
    model = load_checkpoint_into(model, CHECKPOINTS[kind])

    print("Tuning decision threshold on validation split...")
    val_true, val_prob, *_ = run_inference(model, kind, val_loader)
    threshold, val_f1 = select_f1_threshold(val_true, val_prob)
    print(f"{kind}: selected threshold={threshold:.3f} (validation F1={val_f1:.4f})")

    print("Evaluating on test split...")
    test_true, test_prob, test_source_model, test_source_dataset, test_paths = run_inference(model, kind, test_loader)

    metrics = full_metrics(test_true, test_prob, threshold)
    metrics["model"] = kind
    overall_rows.append(metrics)
    print(f"{kind}: test ROC-AUC={metrics['roc_auc']:.4f}  Accuracy={metrics['accuracy']:.4f}  F1(fake)={metrics['f1_fake']:.4f}")

    gen_table = per_generator_table(test_true, test_prob, test_source_model, threshold)
    gen_table["model"] = kind
    per_generator_frames.append(gen_table)

    long_prediction_frames.append(pd.DataFrame({
        "model": kind, "y_true": test_true, "y_prob": test_prob,
        "source_model": test_source_model, "source_dataset": test_source_dataset,
        "relative_path": test_paths,
    }))

    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()


# -----------------------------------------------------------------------------
# 6. Assemble and save comparison tables
# -----------------------------------------------------------------------------

overall_df = pd.DataFrame(overall_rows).set_index("model").reset_index()
overall_df.to_csv(OUTPUT_DIR / "overall_metrics_comparison.csv", index=False)

per_generator_long = pd.concat(per_generator_frames, ignore_index=True)
per_generator_long.to_csv(OUTPUT_DIR / "per_generator_long.csv", index=False)

# Extract consistent test counts per generator
gen_counts = per_generator_long.groupby("generator")["n_fake_images"].first()

# 1. Accuracy Wide Table with sample count
accuracy_wide = per_generator_long.pivot(index="generator", columns="model", values="accuracy_with_reals")
accuracy_wide = accuracy_wide[MODEL_KINDS]
accuracy_wide.insert(0, "n_test_fakes", gen_counts)
accuracy_wide.to_csv(OUTPUT_DIR / "per_generator_accuracy_wide.csv")

# 2. Average Precision Wide Table with sample count
ap_wide = per_generator_long.pivot(index="generator", columns="model", values="average_precision_with_reals")
ap_wide = ap_wide[MODEL_KINDS]
ap_wide.insert(0, "n_test_fakes", gen_counts)
ap_wide.to_csv(OUTPUT_DIR / "per_generator_ap_wide.csv")

# 3. Detection Rate Wide Table with sample count
detection_rate_wide = per_generator_long.pivot(index="generator", columns="model", values="detection_rate")
detection_rate_wide = detection_rate_wide[MODEL_KINDS]
detection_rate_wide.insert(0, "n_test_fakes", gen_counts)
detection_rate_wide.to_csv(OUTPUT_DIR / "per_generator_detection_rate_wide.csv")

full_predictions = pd.concat(long_prediction_frames, ignore_index=True)
full_predictions.to_csv(OUTPUT_DIR / "full_predictions_long.csv", index=False)

# --- Console Outputs ---

print("\n=== Overall metrics, all models ===")
print(
    overall_df[
        [
            "model",
            "roc_auc",
            "average_precision",
            "accuracy",
            "balanced_accuracy",  # Added here
            "f1_fake",
            "cohen_kappa",
            "threshold",
        ]
    ].to_string(index=False)
)

print("\n=== Per-generator accuracy (reals + that generator's fakes) ===")
print(accuracy_wide.to_string())

print("\n=== Per-generator average precision (reals + that generator's fakes) ===")
print(ap_wide.to_string())  # Added print for generator AP

print("\n=== Per-generator detection rate (fakes only) ===")
print(detection_rate_wide.to_string())

print(f"\nAll outputs saved to: {OUTPUT_DIR}")

Device: cuda
Dataset root: /kaggle/input/datasets/nocturnalnerd/balanced-dataset-701020/dataset_result_70_10_20_balanced
Manifest path: /kaggle/input/datasets/nocturnalnerd/balanced-dataset-701020/dataset_result_70_10_20_balanced/manifest.csv


Cloning into '/kaggle/working/FreqNet-DeepfakeDetection'...



Loaded manifest: 55,338 rows, columns: ['split', 'label', 'relative_path', 'source_dataset', 'source_model', 'original_path', 'sha256']
Unique 'split' values: ['train', 'validation', 'test']
Unique 'label' values: ['fake', 'real']
Unique 'source_dataset' values (sample): ['diffusion_defactify', 'gan']
Unique 'source_model' values (sample): ['dalle3', 'midjourney6', 'sd21', 'sd3', 'sdxl', 'AttGAN', 'BEGAN', 'CramerGAN', 'InfoMaxGAN', 'MMDGAN', 'RelGAN', 'S3GAN', 'SNGAN', 'STGAN', 'real']

Path check: 200/200 sampled relative_path values resolve under /kaggle/input/datasets/nocturnalnerd/balanced-dataset-701020/dataset_result_70_10_20_balanced (100.0%)

Validation rows matched: 5,534  |  Test rows matched: 11,068

original_freqnet


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Tuning decision threshold on validation split...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


... (20 outputs left out)

dynamic_freqnet: selected threshold=0.915 (validation F1=0.9713)
Evaluating on test split...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


infer[dynamic_freqnet]:   0%|          | 0/692 [00:00<?, ?it/s]

dynamic_freqnet: test ROC-AUC=0.9958  Accuracy=0.9690  F1(fake)=0.9688

=== Overall metrics, all models ===
           model  roc_auc  average_precision  accuracy  balanced_accuracy  f1_fake  cohen_kappa  threshold
original_freqnet 0.984326           0.984310  0.934134           0.934134 0.934389     0.868269      0.875
      simple_cnn 0.953788           0.955493  0.883628           0.883628 0.882952     0.767257      0.555
 alexnet_scratch 0.955317           0.956508  0.874955           0.874955 0.880152     0.749910      0.295
 dynamic_freqnet 0.995787           0.995100  0.969010           0.969010 0.968804     0.938020      0.915

=== Per-generator accuracy (reals + that generator's fakes) ===
model        n_test_fakes  original_freqnet  simple_cnn  alexnet_scratch  dynamic_freqnet
generator                                                                                
AttGAN                113          0.931114    0.884009         0.829821         0.975739
BEGAN                 